# EMS Data Profiling

Works out what the EPCR (Elite) and outcomes (ESO) data can answer before any analysis starts.

Read only. Nothing is written to any table or view. The one file it produces is an Excel workbook of results at the end.

Everything runs off one SQL statement in section 3. Every analysis cell after that is a short SELECT against it, so the logic is visible rather than buried in code.

## 0. Settings

In [ ]:
CATALOG = "prod"
SCHEMA = "silver_elite_dwgmr"
SCHEMA_ALT = "silver_elite_dwamgh"

YEAR_MIN = 2024
YEAR_MAX = 2026

CALL_TYPE = None
ESO_TABLE = None

BLANKS = "'', 'null', 'none', 'n/a', 'na', 'unknown', 'not recorded', 'not applicable', 'not reporting'"

### What these mean

- `SCHEMA` is the GMR Elite star schema. `SCHEMA_ALT` is the Image Trend side, counted in the inventory only.
- Years start at 2024. Anything earlier was described as inaccurate or absent, so including it would put bad records into every denominator. 2025 is the first year worth quoting externally.
- `CALL_TYPE` is the column that separates 911 scene calls from interfacility transfers. Left empty until section 2 identifies it, then set to a qualified column name such as `sit.Situation_Complaint_Reported_By_Dispatch`.
- `BLANKS` are values that look filled in but are not. NEMSIS writes "Not Recorded" rather than nulls, so a plain null check would call these fields complete.

## 1. What tables exist

In [ ]:
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}").where("databaseName like 'silver%'"))

Lists every silver schema. This is the list that came up when the point was made that the old set was small and the new one has a lot of unfamiliar names in it.

In [ ]:
rows = []
for s in [SCHEMA, SCHEMA_ALT]:
    for t in spark.sql(f"SHOW TABLES IN {CATALOG}.{s}").collect():
        if t.isTemporary:
            continue
        full = f"{CATALOG}.{s}.{t.tableName}"
        try:
            rows.append((s, t.tableName, spark.table(full).count()))
        except Exception:
            rows.append((s, t.tableName, -1))

inventory = spark.createDataFrame(rows, "schema string, table string, rows long").orderBy("rows", ascending=False)
display(inventory)

Row counts for every table in both Elite schemas.

This settles the open question of whether `silver_elite_dwgmr` and `silver_frn_qry_elite_dwgmr_repl` hold the same data. Matching counts confirm the copies are materialized from the same source. A `-1` means the count failed, which in Unity Catalog usually means no read access rather than no table.

## 2. Check the column names

In [ ]:
for t in ["fact_incident", "dim_incident", "dim_situation", "dim_disposition",
          "dim_agency", "dim_patient", "dim_payment", "dim_scene"]:
    print(t)
    print([f.name for f in spark.table(f"{CATALOG}.{SCHEMA}.{t}").schema.fields])
    print()

Prints the columns of the fact table and each dimension.

Run this before section 3. The SQL there names columns explicitly, and if any name is different in your environment this is where you find out. It is also where to look for the secondary impression, the comorbidity fields, and the dispatch complaint field that would fill in `CALL_TYPE`.

## 3. Build the flat dataset

In [ ]:
FLAT = f"""
SELECT
  fi.Incident_Transaction_GUID_Internal          AS incident_id,
  inc.Incident_Date_Time                         AS incident_date,
  pat.Patient_ID_Internal                        AS patient_id,
  pay.Payment_Primary_Method_Of_Payment          AS payer,
  sit.Situation_Provider_Primary_Impression      AS primary_impression,
  inc.Incident_Dispatch_Priority_Patient_Acuity  AS acuity,
  dis.Disposition_Incident_Patient_Disposition   AS disposition,
  sce.Scene_Incident_State_Name                  AS state,
  fi.Incident_Elite_Viewer_State_County_GNIS     AS county,
  dis.Disposition_LZ2_Zip_Code                   AS zip,
  ag.Agency_Name                                 AS agency_name,
  {CALL_TYPE or 'NULL'}                          AS call_type
FROM {CATALOG}.{SCHEMA}.fact_incident fi
LEFT JOIN {CATALOG}.{SCHEMA}.dim_incident    inc ON fi.Dim_Incident_FK    = inc.Dim_Incident_PK
LEFT JOIN {CATALOG}.{SCHEMA}.dim_situation   sit ON fi.Dim_Situation_FK   = sit.Dim_Situation_PK
LEFT JOIN {CATALOG}.{SCHEMA}.dim_disposition dis ON fi.Dim_Disposition_FK = dis.Dim_Disposition_PK
LEFT JOIN {CATALOG}.{SCHEMA}.dim_agency      ag  ON fi.Dim_Agency_FK      = ag.Dim_Agency_PK
LEFT JOIN {CATALOG}.{SCHEMA}.dim_patient     pat ON fi.Dim_Patient_FK     = pat.Dim_Patient_PK
LEFT JOIN {CATALOG}.{SCHEMA}.dim_payment     pay ON fi.Dim_Payment_FK     = pay.Dim_Payment_PK
LEFT JOIN {CATALOG}.{SCHEMA}.dim_scene       sce ON fi.Dim_Scene_FK       = sce.Dim_Scene_PK
"""

print(FLAT)

One row per incident, with the eleven fields the analysis needs.

`fact_incident` is the center of a star schema and holds mostly foreign keys, so the readable values have to be joined in from the dimensions. Each join is `Dim_X_FK` on the fact to `Dim_X_PK` on the dimension.

All joins are LEFT. An incident with no payment row still appears, with a null payer. That is deliberate: a missing dimension row is a finding, and an inner join would hide it.

Two things to check. The column names are written out rather than discovered, so if section 2 shows a different name, edit it here. And the row count should match `fact_incident` exactly. If it is higher, a dimension has duplicate keys, rows have fanned out, and every count below is inflated.

In [ ]:
CALL_CASE = (f"""CASE
      WHEN lower(call_type) RLIKE 'interfacility|inter-facility|ift|transfer' THEN 'Interfacility'
      WHEN lower(call_type) RLIKE '911|emergency|scene|dispatch'              THEN '911 Scene'
      ELSE 'Other/Unclear' END""" if CALL_TYPE else "'Not classified'")

BASE = f"""
WITH flat AS (
{FLAT}
),
scope AS (
  SELECT *,
    year(incident_date)                    AS yr,
    date_format(incident_date, 'yyyy-MM')  AS mo,
    hour(incident_date)                    AS hr,
    date_format(incident_date, 'E')        AS dow,
    CASE
      WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
      WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
      WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
      WHEN payer IS NULL OR lower(trim(payer)) IN ({BLANKS})         THEN 'Unknown'
      ELSE 'Other/Commercial'
    END AS payer_group,
    {CALL_CASE} AS call_origin
  FROM flat
  WHERE year(incident_date) BETWEEN {YEAR_MIN} AND {YEAR_MAX}
),
med AS (SELECT * FROM scope WHERE payer_group = 'Medicaid')
"""

def q(sql):
    return spark.sql(BASE + sql)

def filled(c):
    return f"({c} IS NOT NULL AND lower(trim(cast({c} AS string))) NOT IN ({BLANKS}))"

def completeness(columns, source="scope"):
    parts = [f"""SELECT '{c}' AS field,
                        count(CASE WHEN {filled(c)} THEN 1 END) AS populated,
                        count(*) AS total,
                        round(100.0 * count(CASE WHEN {filled(c)} THEN 1 END) / count(*), 1) AS pct
                 FROM {source}""" for c in columns]
    return q(" UNION ALL ".join(parts) + " ORDER BY pct")

display(q("SELECT count(*) AS incidents, count(DISTINCT incident_id) AS distinct_incidents FROM scope"))

Wraps the join in named steps so every cell below can be a short SELECT.

- `flat` is the join above.
- `scope` adds the date parts, the payer grouping, and the call origin, and cuts to the year window.
- `med` is the Medicaid subset.

These are CTEs inside a single statement, not views or tables. Nothing is created in the catalog.

`q(sql)` runs a SELECT against those steps. `completeness(columns)` measures how many rows have a usable value, treating the NEMSIS placeholders as blank.

The payer grouping in SQL, which is the rule everything Medicaid depends on:

```sql
CASE
  WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
  WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
  WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
  WHEN payer IS NULL OR lower(trim(payer)) IN (...)              THEN 'Unknown'
  ELSE 'Other/Commercial'
END
```

First match wins. The output of this cell is the fan-out check: incidents against distinct incident ids. If they differ, the join duplicated rows.

## 4. Volume

In [ ]:
display(q("SELECT yr, count(*) AS incidents FROM scope GROUP BY yr ORDER BY yr"))

Counts by year. A year that looks far too small is partial coverage, not a change in demand.

In [ ]:
display(q("SELECT mo, count(*) AS incidents FROM scope GROUP BY mo ORDER BY mo"))

Monthly counts. Look for a ramp at the start and a drop at the end; both are coverage artifacts.

In [ ]:
display(q("""
    SELECT yr, agency_name, count(*) AS incidents
    FROM scope GROUP BY yr, agency_name
    ORDER BY yr, incidents DESC
"""))

Counts by agency and year. An agency onboarding mid-period looks like growth; one dropping out looks like decline. Neither is about patient demand.

In [ ]:
display(q("SELECT state, count(*) AS incidents FROM scope GROUP BY state ORDER BY incidents DESC"))

The geographic footprint. Medicaid programs are state-run, so a national figure built on a handful of states needs that said out loud.

## 5. Completeness

In [ ]:
FIELDS = ["incident_id", "incident_date", "patient_id", "payer", "primary_impression",
          "acuity", "disposition", "state", "county", "zip", "agency_name"]

display(completeness(FIELDS))

How many incidents have a usable value in each field. Blank counts nulls and the NEMSIS placeholders.

This is the direct test of what was described in the meeting: roughly half the patients present, but only around a tenth of the information filled in.

In [ ]:
parts = []
for c in FIELDS:
    parts.append(f"""SELECT '{c}' AS field, yr,
                            round(100.0 * count(CASE WHEN {filled(c)} THEN 1 END) / count(*), 1) AS pct
                     FROM scope GROUP BY yr""")

completeness_by_year = q(" UNION ALL ".join(parts) + " ORDER BY field, yr")
display(completeness_by_year)

The same percentages by year. A field well populated in older years and empty recently means a workflow or vendor feed changed, which matters more than the average.

## 6. Payer mix

In [ ]:
payer_values = q("""
    SELECT payer, count(*) AS incidents,
           round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
    FROM scope GROUP BY payer ORDER BY incidents DESC LIMIT 50
""")
display(payer_values)

Every distinct payer value before grouping.

Read this first. Medicaid is flagged explicitly about 4.4% of the time while a generic "Insurance" value takes 38%, so the Medicaid figure is a floor rather than an estimate until that bucket is understood.

In [ ]:
payer_mix = q("""
    SELECT yr, payer_group, count(*) AS incidents
    FROM scope GROUP BY yr, payer_group ORDER BY yr, incidents DESC
""")
display(payer_mix)

The grouped version by year. This is the Medicaid share of EMS volume.

Watch Unknown. If it is large or growing in the most recent year, the Medicaid share is understated by an amount we cannot measure. A rough Genie query put Medicaid near 230,000 patients last year, which is a useful cross-check against what this returns for 2025.

In [ ]:
display(spark.sql(f"""
    SELECT * FROM {CATALOG}.{SCHEMA}.dim_payment LIMIT 20
"""))

Twenty rows straight from the payment dimension, restricted to nothing.

This is the fastest way to answer the open question about the Insurance bucket. Look for a secondary method, a plan name, or a company field that carries the real payer. If Medicaid managed care plan names appear there, widening the Medicaid branch of the CASE in section 3 to match those names is a one line change that moves the share materially.

In [ ]:
payer_by_county = q("""
    SELECT state, county, payer_group, count(*) AS incidents
    FROM scope GROUP BY state, county, payer_group
    ORDER BY incidents DESC LIMIT 200
""")
display(payer_by_county)

County is the level payer conversations happen at, and the level ACS, HPSA and CDC PLACES join at. This is the cut that supports both.

In [ ]:
call_origin_mix = q("""
    SELECT yr, payer_group, call_origin, count(*) AS incidents
    FROM scope GROUP BY yr, payer_group, call_origin
    ORDER BY yr, incidents DESC
""")
display(call_origin_mix)

911 scene calls against interfacility transfers.

The two uses pull opposite ways. For a payer or customer view, transfers are legitimate volume and belong in. For the question being chased, whether a Medicaid patient arriving through a 911 dispatch could be routed differently, a scheduled hospital-to-hospital transfer is not that patient and does not belong in the denominator.

This returns "Not classified" until `CALL_TYPE` is set in section 0. Use section 2 to find the dispatch complaint or service requested column, set it, and re-run from section 3.

## 7. How often patients come back

In [ ]:
display(q("""
    SELECT count(*) AS incidents,
           count(DISTINCT patient_id) AS distinct_patients,
           round(count(*) / count(DISTINCT patient_id), 2) AS incidents_per_patient
    FROM med
"""))

Run this before reading anything below it.

If incidents per patient comes back near 1.0, the identifier is generated per encounter rather than per person. The tiers and the return intervals below would then be measuring nothing, and repeat utilization cannot be built from this table at all.

That is a finding in its own right, since the utilization pyramid and the recurrent demand model are both named as high value products.

In [ ]:
medicaid_pyramid = q("""
    WITH per_patient AS (
      SELECT patient_id, count(*) AS encounters
      FROM med WHERE patient_id IS NOT NULL
      GROUP BY patient_id
    )
    SELECT
      CASE WHEN encounters = 1 THEN '1'
           WHEN encounters <= 4 THEN '2-4'
           WHEN encounters <= 11 THEN '5-11'
           ELSE '12+' END AS tier,
      count(*) AS patients,
      sum(encounters) AS encounters,
      round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct_patients,
      round(100.0 * sum(encounters) / sum(sum(encounters)) OVER (), 2) AS pct_encounters
    FROM per_patient GROUP BY 1 ORDER BY 1
""")
display(medicaid_pyramid)

Medicaid patients grouped into 1, 2-4, 5-11 and 12 or more encounters.

The point is the gap between the two percentage columns. National work shows a small group of high utilizers driving a disproportionate share of encounters, and those two columns side by side are what make that visible.

Only meaningful if the previous cell showed a patient id that persists across visits.

In [ ]:
display(q("""
    WITH gaps AS (
      SELECT datediff(
               lead(incident_date) OVER (PARTITION BY patient_id ORDER BY incident_date),
               incident_date) AS days_to_next
      FROM med WHERE patient_id IS NOT NULL
    )
    SELECT count(*) AS repeat_pairs,
           percentile_approx(days_to_next, 0.5) AS median_days,
           round(100.0 * avg(CASE WHEN days_to_next <= 7 THEN 1 ELSE 0 END), 1) AS pct_within_7d,
           round(100.0 * avg(CASE WHEN days_to_next <= 30 THEN 1 ELSE 0 END), 1) AS pct_within_30d
    FROM gaps WHERE days_to_next IS NOT NULL
""")) 

Time to the next encounter for the same patient, using `lead()`. The 7 and 30 day windows are the two named in the recurrent demand model. Same dependency on the patient id as above.

## 8. Clinical and timing

In [ ]:
top_impressions = q("""
    SELECT primary_impression, count(*) AS incidents,
           round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
    FROM med GROUP BY primary_impression ORDER BY incidents DESC LIMIT 30
""")
display(top_impressions)

The most common primary impressions among Medicaid incidents, which are the primary ICD-10 codes on those records.

This is the input to any avoidable episode work. Treating everything here as low acuity would be wrong, which is why the actual distribution matters.

In [ ]:
behavioral_health = q("""
    SELECT yr, count(*) AS medicaid_incidents,
           sum(CASE WHEN lower(primary_impression) RLIKE
               'behavioral|psychiatric|anxiety|depress|bipolar|schizo|suicide|substance|alcohol|overdose'
               THEN 1 ELSE 0 END) AS behavioral,
           round(100.0 * avg(CASE WHEN lower(primary_impression) RLIKE
               'behavioral|psychiatric|anxiety|depress|bipolar|schizo|suicide|substance|alcohol|overdose'
               THEN 1 ELSE 0 END), 1) AS pct
    FROM med GROUP BY yr ORDER BY yr
""")
display(behavioral_health)

Behavioral health share of Medicaid EMS, measured on coded impressions.

This is the EPCR side counterpart to the nurse navigation note screening. That work reads free text nurse notes; this reads coded impressions. Two independent measurements of the same population are worth considerably more than either one alone.

Matching on full words here, not fragments, which avoids the problem where a short acronym matches inside an unrelated word.

In [ ]:
hour_by_day = q("SELECT dow, hr, count(*) AS incidents FROM med GROUP BY dow, hr ORDER BY dow, hr")
display(hour_by_day)

Demand by day and hour. The claim to test is that weekday afternoons carry the heaviest volume while overnight has the worst response times, which would mean staffing built around a morning peak is aimed at the wrong hours. Days sort alphabetically, so read the labels.

In [ ]:
display(q("""
    SELECT disposition, count(*) AS incidents
    FROM med GROUP BY disposition ORDER BY incidents DESC LIMIT 30
"""))

Transported, treated and released, refused. This separates billable transports from unpaid work. Check the completeness of this field before reading much into the distribution.

In [ ]:
display(q("""
    SELECT acuity, payer_group, count(*) AS incidents
    FROM scope GROUP BY acuity, payer_group ORDER BY incidents DESC
"""))

Acuity against payer group. Relevant to the claim that Medicaid EMS demand is not simply low acuity misuse. If the Medicaid distribution resembles the other payer groups, that claim has support in our own data.

## 9. ESO outcomes

In [ ]:
where = " OR ".join([f"lower(table_schema) LIKE '%{k}%' OR lower(table_name) LIKE '%{k}%'"
                     for k in ["eso", "outcome", "hospital", "discharge"]])

candidates = spark.sql(f"""
    SELECT table_catalog, table_schema, table_name
    FROM system.information_schema.tables
    WHERE {where}
    ORDER BY table_catalog, table_schema, table_name
""")
display(candidates)

The schema we were pointed to was empty, so this searches the metadata catalog for anything outcomes related.

Metadata search works even where the data itself is not readable, so this returns results either way. If nothing here is readable, that is permissions rather than naming, since Unity Catalog reports objects you cannot read as missing.

In [ ]:
eso_inventory = []
for r in candidates.limit(100).collect():
    full = f"{r.table_catalog}.{r.table_schema}.{r.table_name}"
    try:
        eso_inventory.append((full, spark.table(full).count(), "ok"))
    except Exception as e:
        eso_inventory.append((full, -1, str(e)[:120]))

eso_inventory.sort(key=lambda x: -x[1])
display(spark.createDataFrame(eso_inventory, "table string, rows long, status string"))

Tries to count each candidate and keeps the real error rather than hiding it. The status column is what tells you whether this is an access problem or a missing data problem.

In [ ]:
readable = [r[0] for r in eso_inventory if r[2] == "ok"]
if ESO_TABLE is None and readable:
    ESO_TABLE = readable[0]

print(ESO_TABLE)
if ESO_TABLE:
    print([f.name for f in spark.table(ESO_TABLE).schema.fields])

Picks the largest readable table, or whatever is pinned in section 0, and prints its columns. If nothing is readable the rest of this section skips and the run continues.

In [ ]:
eso_match = None
if ESO_TABLE:
    ESO_KEY = "Incident_Number"
    eso_match = spark.sql(BASE + f"""
        SELECT s.yr,
               count(*) AS epcr_records,
               count(e.k) AS with_outcome,
               round(100.0 * count(e.k) / count(*), 1) AS pct_matched
        FROM scope s
        LEFT JOIN (SELECT DISTINCT cast({ESO_KEY} AS string) AS k FROM {ESO_TABLE}) e
          ON cast(s.incident_id AS string) = e.k
        GROUP BY s.yr ORDER BY s.yr
    """)
    display(eso_match)

The share of EPCR incidents with a matching outcome record, by year.

Set `ESO_KEY` to the column from the print above that actually corresponds to the Elite transaction GUID. There is no documented key between the two systems, so this is a deliberate choice, not a guess to leave alone.

The `SELECT DISTINCT` matters: without it, a patient with several outcome rows would duplicate the EPCR row and inflate the count.

Zero matches means the wrong key, not missing data. A real coverage gap gives a small number, not none.

In [ ]:
if eso_match is not None:
    display(spark.sql(BASE + f"""
        SELECT s.acuity,
               count(*) AS n,
               round(100.0 * count(e.k) / count(*), 1) AS pct_matched
        FROM scope s
        LEFT JOIN (SELECT DISTINCT cast({ESO_KEY} AS string) AS k FROM {ESO_TABLE}) e
          ON cast(s.incident_id AS string) = e.k
        GROUP BY s.acuity ORDER BY n DESC
    """))

The same match rate broken out by acuity. This is the bias test.

Hospitals were reported as not sending low acuity records. If the match rate is high for serious cases and near zero for minor ones, the matched set is not a sample of our patients, it is a sample of our sickest patients, and no outcome rate computed from it generalizes.

## 10. Geography

In [ ]:
geo_completeness = completeness(["state", "county", "zip"], source="med")
display(geo_completeness)

Which geography fields are populated on Medicaid incidents. Whichever holds up sets the join level for public data. County is usually the realistic answer.

## 11. What we can answer

In [ ]:
QUESTIONS = {
    "Medicaid share of EMS volume": ["payer", "incident_date"],
    "Medicaid mix by county": ["county", "payer"],
    "911 scene vs interfacility split": ["call_type", "payer"],
    "Behavioral health share of Medicaid EMS": ["primary_impression", "payer"],
    "Utilization pyramid (1 / 2-4 / 5-11 / 12+)": ["patient_id", "payer"],
    "7- and 30-day recurrent demand": ["patient_id", "incident_date"],
    "Clinical mix / potentially avoidable episodes": ["primary_impression", "payer"],
    "Time-of-day and day-of-week demand": ["incident_date"],
    "Transport vs non-transport disposition": ["disposition", "payer"],
    "Acuity mix by payer": ["acuity", "payer"],
    "Geographic overlay with ACS / HPSA / PLACES": ["county", "state"],
    "ED outcome linkage": ["incident_id"],
}

pct = {r.field: r.pct for r in completeness(
    sorted({f for v in QUESTIONS.values() for f in v}), source="med").collect()}

rows = []
for question, fields in QUESTIONS.items():
    worst = min(pct.get(f, 0.0) for f in fields)
    verdict = "Ready" if worst >= 80 else ("Caveat needed" if worst >= 40 else "Blocked")
    rows.append((question, ", ".join(fields), worst, verdict))

scorecard = spark.createDataFrame(rows, "question string, fields string, weakest_pct double, verdict string")                  .orderBy("weakest_pct", ascending=False)
display(scorecard)

Each question scored by the least complete field it needs. 80 or above is Ready, 40 to 80 needs a caveat, below 40 is blocked.

This is the draft list to take back to Noah and Rex: here is what the data can answer, you tell us which ones matter and what you would do with the answer. Scoring them means the conversation starts from what is possible rather than from a general offer to analyze.

One limitation to state whenever this is shown. It measures whether fields are filled in, not whether they are correct or usable. A patient id populated on every row still scores 100 even if it is regenerated each encounter. The outcome linkage row only checks that the EPCR key exists, not that anything matches it. This flags empty fields; it cannot flag fields full of the wrong thing.

## 12. Save results

In [ ]:
import os
import pandas as pd

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
RESULTS = os.path.join("/Workspace", os.path.dirname(nb_path).lstrip("/"), "results")
os.makedirs(RESULTS, exist_ok=True)

sheets = {
    "table_inventory": inventory,
    "completeness": completeness(FIELDS),
    "completeness_by_year": completeness_by_year,
    "payer_values": payer_values,
    "payer_mix": payer_mix,
    "payer_by_county": payer_by_county,
    "call_origin_mix": call_origin_mix,
    "medicaid_pyramid": medicaid_pyramid,
    "top_impressions": top_impressions,
    "behavioral_health": behavioral_health,
    "hour_by_day": hour_by_day,
    "geo_completeness": geo_completeness,
    "scorecard": scorecard,
}

if eso_match is not None:
    sheets["eso_match_by_year"] = eso_match

path = os.path.join(RESULTS, "ems_data_profiling_results.xlsx")
with pd.ExcelWriter(path, engine="openpyxl") as writer:
    for name, df in sheets.items():
        df.limit(5000).toPandas().to_excel(writer, sheet_name=name[:31], index=False)

print(path)

One workbook into a `results` folder beside this notebook. A file, not a table.

Each sheet caps at 5,000 rows; the hour by day grid and county cuts are the ones that could hit it.

This is also the de-identified extract that was asked for. Counts and rates only, no patient names, so it can be shared without a data access request.

## 13. Open questions

- What is inside the generic Insurance value on the payer field? Section 6 goes at it. Until that is answered every Medicaid figure here is a floor, not an estimate.
- Do we keep or exclude interfacility transfers? Different answer for a payer view than for the 911 routing question. Set `CALL_TYPE` and both are available.
- Does the patient id stay the same across incidents, or get created fresh each call? The first cell of section 7 answers it, and all the repeat use work depends on it.
- Does the payer field come from billing, or from the crew at the scene? Revenue cycle data is expected in about three weeks and should settle it.
- Which column links EPCR to the outcomes feed? Zero matches means the wrong key, not missing data.
- Where do the nurse navigation calls and the EPCR records overlap? The link is patients who were transported, since those appear in both. That join is the basis for measuring whether a routing decision worked.
- Image Trend (`silver_elite_dwamgh`) access, and whether it needs adding for national coverage.